In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('spam.csv', encoding='latin-1')
df.shape

(5572, 5)

In [3]:
df = df[['v1', 'v2']]
df.columns = ['label', 'message']
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
df['label'].value_counts()

,count
label,
ham,4825
spam,747


In [5]:
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})
df.head()

,label,message,label_num
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['message'], df['label_num'], test_size=0.2, random_state=42, stratify=df['label_num']
)

print(X_train.shape, X_test.shape)

(4457,) (1115,)


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english', max_features=3000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)   # transform only, not fit — reuse train's vocabulary

print(X_train_tfidf.shape, X_test_tfidf.shape)

(4457, 3000) (1115, 3000)


In [8]:
print(tfidf.get_feature_names_out()[:20])

['00' '000' '02' '0207' '03' '04' '05' '06' '07' '07123456789'
 '07742676969' '07781482378' '07821230901' '07xxxxxxxxx' '0800'
 '08000839402' '08000930705' '08000938767' '08001950382' '08002986030']


In [9]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)

print("Naive Bayes trained.")

Naive Bayes trained.


In [10]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_tfidf, y_train)

print("Logistic Regression trained.")

Logistic Regression trained.


In [11]:
from sklearn.svm import SVC

svm = SVC(kernel='linear', probability=True, random_state=42)
svm.fit(X_train_tfidf, y_train)

print("SVM trained.")

SVM trained.


In [12]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred_nb = nb.predict(X_test_tfidf)

print("=== Naive Bayes ===")
print(classification_report(y_test, y_pred_nb))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))
print("ROC-AUC:", roc_auc_score(y_test, nb.predict_proba(X_test_tfidf)[:, 1]))

=== Naive Bayes ===
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       966
           1       0.98      0.81      0.89       149

    accuracy                           0.97      1115
   macro avg       0.98      0.91      0.94      1115
weighted avg       0.97      0.97      0.97      1115

Confusion Matrix:
[[964   2]
 [ 28 121]]
ROC-AUC: 0.9889775869495742


In [13]:
y_pred_lr = log_reg.predict(X_test_tfidf)

print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred_lr))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, log_reg.predict_proba(X_test_tfidf)[:, 1]))

=== Logistic Regression ===
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       966
           1       1.00      0.79      0.88       149

    accuracy                           0.97      1115
   macro avg       0.98      0.90      0.93      1115
weighted avg       0.97      0.97      0.97      1115

Confusion Matrix:
[[966   0]
 [ 31 118]]
ROC-AUC: 0.9863235927577918


In [14]:
y_pred_svm = svm.predict(X_test_tfidf)

print("=== SVM ===")
print(classification_report(y_test, y_pred_svm))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))
print("ROC-AUC:", roc_auc_score(y_test, svm.predict_proba(X_test_tfidf)[:, 1]))

=== SVM ===
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       966
           1       0.98      0.88      0.93       149

    accuracy                           0.98      1115
   macro avg       0.98      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115

Confusion Matrix:
[[964   2]
 [ 18 131]]
ROC-AUC: 0.9852953436991955


In [15]:
sample_messages = [
    "Congratulations! You've won a $1000 gift card. Click here to claim now!",
    "Hey, are we still meeting for lunch tomorrow?",
    "URGENT: Your account has been suspended. Verify immediately at this link.",
    "Can you send me the notes from today's class?"
]

sample_tfidf = tfidf.transform(sample_messages)
predictions = svm.predict(sample_tfidf)

for msg, pred in zip(sample_messages, predictions):
    label = "SPAM" if pred == 1 else "HAM (legitimate)"
    print(f"[{label}] {msg}")

[SPAM] Congratulations! You've won a $1000 gift card. Click here to claim now!
[HAM (legitimate)] Hey, are we still meeting for lunch tomorrow?
[SPAM] URGENT: Your account has been suspended. Verify immediately at this link.
[HAM (legitimate)] Can you send me the notes from today's class?
